# Gravitino Virtual File System (GVFS): governed access to unstructured data

Structured data lives in tables; a lot of the data that matters, documents, images, model artifacts, lives in files. Gravitino manages those files as **filesets**, and the **Gravitino Virtual File System (GVFS)** is how you *read and write* their contents through Gravitino rather than through a raw storage path.

Why that matters: with GVFS, an application opens a path like `fileset/catalog/schema/fileset_name/...` and Gravitino resolves it to the real storage location, applying the same governance, cataloging, and (with auth enabled) authorization that it applies to tables. The application never hardcodes an S3/HDFS/local path, and access goes through the governed layer.

This notebook does one thing: point a fileset at a folder of PDFs, then read those files **through GVFS**, so you can see governed unstructured-data access on its own, with no LLM or RAG involved. (The LlamaIndex demo shows the same GVFS step feeding a RAG pipeline; here it stands alone.)

## Install the client

The Gravitino Python client includes the `gvfs` module. `fsspec` provides the filesystem interface GVFS implements.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q apache-gravitino==1.3.0 fsspec

## Register a fileset over the demo documents

The playground ships a few city PDFs at `/tmp/gravitino/data/pdfs` inside the Gravitino container. We register a fileset pointing at that folder: catalog `catalog_fileset`, schema `countries`, fileset `cities`. Each step is load-or-create, so the notebook is safe to re-run.

In [ ]:
from gravitino import NameIdentifier, GravitinoClient, Catalog, Fileset

gravitino_url = "http://gravitino:8090"
metalake_name = "metalake_demo"
catalog_name  = "catalog_fileset"
schema_name   = "countries"
fileset_name  = "cities"
fileset_ident = NameIdentifier.of(schema_name, fileset_name)

client = GravitinoClient(uri=gravitino_url, metalake_name=metalake_name)

# Catalog (fileset type, hadoop provider).
try:
    catalog = client.load_catalog(name=catalog_name)
    print(f"catalog {catalog_name}: loaded")
except Exception:
    catalog = client.create_catalog(
        name=catalog_name, catalog_type=Catalog.Type.FILESET,
        comment="demo", provider="hadoop", properties={})
    print(f"catalog {catalog_name}: created")

# Schema.
try:
    catalog.as_schemas().load_schema(schema_name=schema_name)
    print(f"schema {schema_name}: loaded")
except Exception:
    catalog.as_schemas().create_schema(schema_name=schema_name, comment="countries", properties={})
    print(f"schema {schema_name}: created")

# Fileset pointing at the PDFs on disk (external: Gravitino tracks it, does not own the storage).
try:
    fs_obj = catalog.as_fileset_catalog().load_fileset(ident=fileset_ident)
    print(f"fileset {fileset_name}: loaded")
except Exception:
    fs_obj = catalog.as_fileset_catalog().create_fileset(
        ident=fileset_ident,
        fileset_type=Fileset.Type.EXTERNAL,
        comment="cities",
        storage_location="file:/tmp/gravitino/data/pdfs",
        properties={})
    print(f"fileset {fileset_name}: created")

print("\nfileset storage location:", fs_obj.storage_location())

## Open the Gravitino Virtual File System

`GravitinoVirtualFileSystem` is an `fsspec`-compatible filesystem. You give it the Gravitino server and metalake; from then on you address files by their **virtual** path, `fileset/{catalog}/{schema}/{fileset}/...`, and GVFS resolves each one to the real storage location behind the scenes.

In [ ]:
from gravitino import gvfs

fs = gvfs.GravitinoVirtualFileSystem(
    server_uri=gravitino_url,
    metalake_name=metalake_name,
)

# The virtual root of our fileset. Note: no storage path (no file:/, hdfs://, s3://)
# appears here. The application only knows the governed fileset path.
fileset_root = f"fileset/{catalog_name}/{schema_name}/{fileset_name}"
print("virtual root:", fileset_root)

## List the files through GVFS

Listing the virtual path returns the documents in the fileset, resolved through Gravitino. This is the governed equivalent of `ls` on the folder.

In [ ]:
entries = fs.ls(fileset_root)
print(f"{len(entries)} file(s) in the fileset:\n")
for e in entries:
    # fsspec entries may be dicts (with 'name'/'size') or plain path strings.
    if isinstance(e, dict):
        print(f"  {e.get('name')}  ({e.get('size', '?')} bytes)")
    else:
        print(f"  {e}")

## Read a file's contents through GVFS

Now the point of the whole exercise: open one of the documents **by its virtual path** and read its bytes. The application never touches the underlying `file:/tmp/gravitino/data/pdfs` location, GVFS resolves it. Swap the backend to S3 or HDFS later and this code does not change.

In [ ]:
# Pick the first file from the listing.
first = entries[0]
first_path = first['name'] if isinstance(first, dict) else first
# Normalize to a virtual path GVFS understands.
if not first_path.startswith('fileset/'):
    first_path = f"{fileset_root}/{first_path.split('/')[-1]}"

with fs.open(first_path, 'rb') as f:
    data = f.read()

print(f"read {len(data):,} bytes from {first_path} via GVFS")
# PDFs start with the magic bytes b'%PDF'. Show it to prove we read real content.
print("first bytes:", data[:8])
print("looks like a PDF:", data[:4] == b"%PDF")

## What just happened, and why it matters

Every read above went `application -> GVFS -> Gravitino -> real storage`. The application code only ever named a governed path (`fileset/catalog_fileset/countries/cities/...`); it never knew or hardcoded where the bytes actually live. That indirection is the whole value:

- **Portability**: the same code reads the fileset whether it is backed by local disk, HDFS, S3, or GCS. Change the fileset's `storage_location`, not your application.
- **Governance**: access to unstructured data flows through the same catalog as your tables, so it can be discovered, tagged, and audited alongside everything else.
- **Authorization**: with authentication enabled, GVFS access is authorized per principal, the same governance you get on structured data, extended to files.

That is how Gravitino provides **unstructured** data: not as loose paths scattered across buckets, but as governed filesets read through a virtual filesystem. The LlamaIndex demo notebook takes this same GVFS step and feeds the documents into a RAG pipeline; everything above is the governed data-access foundation underneath it.